# 🔁 Module 1.7 — Session State and Resumption

**Domain 1 · Agentic Architecture & Orchestration** (27% of the exam) — **final module in this domain**
**Task 1.7 · Session State and Resumption**
**Source:** [claudecertificationguide.com/learn/1-agentic-architecture/1-7-session-state-resumption](https://claudecertificationguide.com/learn/1-agentic-architecture/1-7-session-state-resumption)

Back to the real Claude Agent SDK (Modules 1.3/1.5's territory) — this
module is about session mechanics the raw Messages API simply doesn't have
a concept of at all. The Messages API is stateless by design (you always
send the full `messages` list yourself); *sessions* — named, resumable,
forkable, persisted to disk — are a Claude Code / Agent SDK feature.

### 🎯 What you'll build

A real named session that analyzes 10 files, a real fork of it, a real
demonstration of the **stale context problem** (resuming after files change
gives contradictory advice), and the real fix (a fresh session with an
injected summary) — compared side by side.

### ✅ What you'll walk away knowing

1. The three session strategies — `--resume`, `fork_session`, fresh start
   with summary injection — and exactly when each one is right
2. Why resuming after file changes causes contradictory advice (the stale
   context problem), concretely, not just in theory
3. Why re-reading the changed files isn't enough on its own — the naive fix,
   and why it still fails
4. Why `fork_session` is for exploring divergent approaches, never for
   fixing stale context
5. Targeted re-analysis (tell it what changed) vs. wasteful full re-exploration

---

> **💡 A pleasant surprise while building this one:** `fork_session`,
> `rename_session`, `list_sessions`, and `get_session_messages` all turned
> out to be **local file operations** on a session's JSONL transcript —
> verified by reading their docstrings on the installed package — not API
> calls. So most of this notebook's session mechanics cost nothing at all;
> only the actual *analysis* steps (Tasks 1, 4, 5) touch the real API.
>
> **💳 Cost:** 3 real API calls total (initial analysis, the stale-resume
> attempt, the fresh-start fix) — everything else (naming, forking, listing,
> reading transcripts) is free, local, and instant.

## 🔧 Setup

Same package as Modules 1.3/1.5.

```bash
pip install claude-agent-sdk
```

```bash
# Windows (PowerShell)
$env:ANTHROPIC_API_KEY = "sk-ant-..."

# macOS / Linux (bash/zsh)
export ANTHROPIC_API_KEY="sk-ant-..."
```


In [ ]:
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Set it in your shell, then restart the "
        "kernel and run this cell again -- see the Setup section above."
    )

from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    rename_session,
    fork_session,
    list_sessions,
    get_session_info,
    get_session_messages,
)

print("claude_agent_sdk imported. Ready.")


## 🔑 Key Concept: Three Session Management Options

### 1. `--resume <session>`
Restores a session from exactly where it stopped — full conversation
history, every tool result, every prior analysis. **Use when:** prior
context is still valid and files haven't changed. **Don't use when:** files
have been modified since — that's the stale context problem, below.

### 2. `fork_session`
Branches an **independent copy** from a shared baseline. Each branch
proceeds on its own; neither can see the other's later results. **Verified
real SDK shape:** fork is a modifier passed *alongside* `resume`, not an
alternative to it — `resume` names the source, `fork_session=True` says
*branch* instead of *append*. **Use when:** comparing divergent approaches
from the same starting point. **Don't use when:** you're just continuing
one line of work (that's plain `resume`) — and never as a fix for stale
context, since a fork inherits the exact same stale history it branched from.

### 3. Fresh start with summary injection
A brand-new session, with a structured summary of prior findings typed
directly into the first prompt. No stale tool results ride along, because
there's no history to inherit at all. **Use when:** files have changed, or a
long session's context has degraded into clutter.

### Decision matrix

| Scenario | Best option | Why |
|---|---|---|
| Continuing yesterday's work, nothing changed | `--resume` | Prior context is still valid |
| Comparing two refactoring approaches | `fork_session` | Divergent exploration, same baseline |
| Resuming after modifying 3 of 50 files | Fresh start + summary | Stale results for the changed files would contradict reality |
| Long session, cluttered history | Fresh start + summary | A curated summary beats a degraded transcript |
| Exploring testing strategy vs. documentation strategy | `fork_session` | Two independent directions from one analysis |
| Resuming after a dependency update | Fresh start + summary | Indirect effects could touch files you didn't expect |


## 🔑 Key Concept: The Stale Context Problem

**What happens:** you resume a session after modifying code, and the agent
reasons from **cached tool results** that no longer reflect reality — old
`Read` results are still sitting in the restored conversation history right
alongside anything new.

**Why it happens, precisely:** resuming restores the *entire* history,
tool results included. If a file was read last session and has since
changed on disk, the old contents are still there, as a message, exactly as
they were. The model doesn't get to choose to forget them.

### The naive fix, and why it's not enough

*"Just resume and ask it to re-read the changed files."* The **new** read is
correct — but the **old**, now-contradictory read is still sitting
earlier in the same history. For anything not directly about the three
changed files, the model may still lean on the stale version alongside the
fresh one, because both are technically "in context."

### The actual fix

Start a **fresh** session. Inject a structured summary of what was already
found. Explicitly name which files changed, so the agent does **targeted
re-analysis** of just those — not a full re-exploration of everything, and
not a resume that drags the old reads along for the ride.


## 📖 Case Study: The Contradictory Advice Bug

A developer analyzes a 50-file codebase over two days:

- **Day 1:** analyzes the auth module, finds three issues.
- **Overnight:** fixes all three, editing `auth.ts`, `session.ts`, `middleware.ts`.
- **Day 2:** resumes the session.

**What happens:** Claude recommends fixing issues that are already fixed,
and gives **inconsistent** answers about the current state of `auth.ts` —
sometimes describing the old code (from the stale tool result), sometimes
the new (if it happens to re-read). Same file, same session, contradicting
itself.

**The fix:** a fresh session, told directly:

> "Prior analysis identified three authentication issues in auth.ts,
> session.ts, and middleware.ts. All three have been fixed. Please
> re-analyse these three files to verify the fixes and check for any new
> issues introduced by the changes."

No stale results to contradict anything — just a clean read of what's
actually there now, informed by what was already learned.

This is exactly what Tasks 1–6 below build and compare, for real.


## 🛠️ Build Exercise — Task 1: Named Session, 10-File Analysis

**Objective:** a real session analyzing 10 real files on disk, given a
memorable name.

**Why this matters:** naming makes a session something you can deliberately
come back to later — which is exactly what Tasks 4 and 5 will each do, in
two different ways.

This cell writes 10 real files to a local scratch directory, then makes 1
real API call.


In [ ]:
import tempfile
from pathlib import Path

DEMO_DIR = Path(tempfile.gettempdir()) / "ccar_session_demo_codebase"
DEMO_DIR.mkdir(exist_ok=True)

# Ten small files, three of them carrying a real, fixable null-pointer risk
# (unguarded property chains) -- these three are exactly what Task 3 will
# modify for real, later.
FILES = {
    "file_01.js": "function formatCurrency(amount) {\n  return `$${amount.toFixed(2)}`;\n}",
    "file_02.js": "function slugify(text) {\n  return text.toLowerCase().trim().replace(/\\s+/g, '-');\n}",
    "file_03.js": "function sumValues(items) {\n  let total = 0;\n  items.forEach(function (item) {\n    total += item.value;\n  });\n  return total;\n}",
    "file_04.js": "function isValidEmail(email) {\n  return /^[^\\s@]+@[^\\s@]+\\.[^\\s@]+$/.test(email);\n}",
    "file_05.js": "function getUserTheme(req) {\n  return req.user.profile.settings.theme;\n}",
    "file_06.js": "function chunkArray(arr, size) {\n  const chunks = [];\n  for (let i = 0; i < arr.length; i += size) {\n    chunks.push(arr.slice(i, i + size));\n  }\n  return chunks;\n}",
    "file_07.js": "function findUserById(db, userId) {\n  const query = `SELECT * FROM users WHERE id = ${userId}`;\n  return db.execute(query);\n}",
    "file_08.js": "function getShippingCity(order) {\n  return order.customer.address.shipping.city;\n}",
    "file_09.js": "function logEvent(name, payload) {\n  console.log(`[event] ${name}`, payload);\n}",
    "file_10.js": "function getInvoiceContact(invoice) {\n  return invoice.billing.contact.email.address;\n}",
}

for name, content in FILES.items():
    (DEMO_DIR / name).write_text(content, encoding="utf-8")

print(f"Wrote {len(FILES)} files to {DEMO_DIR}")

session_options = ClaudeAgentOptions(
    cwd=str(DEMO_DIR),
    allowed_tools=["Read", "Glob"],
)

ANALYSIS_PROMPT = (
    "Analyze all the JavaScript files in this directory. For each file, "
    "note any bugs, security issues, or code quality concerns you find."
)


async def run_initial_analysis():
    session_id = None
    final_text = None
    async for message in query(prompt=ANALYSIS_PROMPT, options=session_options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"[assistant] {block.text[:200]}")
        elif isinstance(message, ResultMessage):
            session_id = message.session_id
            final_text = message.result
            if message.total_cost_usd is not None:
                print(f"Cost: ${message.total_cost_usd:.4f}")
    return session_id, final_text


initial_session_id, initial_analysis_text = await run_initial_analysis()

print()
print("Session ID:", initial_session_id)
print()
print("=== Final analysis ===")
print(initial_analysis_text)

# Naming -- a FREE, local operation (appends a title entry to the session's
# JSONL transcript; no API call involved). Wrapped defensively: the CLI
# writes the transcript file asynchronously, so calling this an instant
# after query() returns can occasionally race a not-yet-flushed file.
try:
    rename_session(initial_session_id, "auth-module-analysis")
    info = get_session_info(initial_session_id, directory=str(DEMO_DIR))
    print()
    print("Session title is now:", info.custom_title if info else "(session file not found)")
except FileNotFoundError as e:
    print()
    print(f"Could not rename the session yet: {e}")
    print("The transcript file may not be flushed to disk yet -- re-run this cell.")


### A free, real look at `fork_session`

Before moving on to the stale-context scenario, here's `fork_session` doing
its actual job — branching, not continuing. This costs nothing (it's a
local file copy, verified from the installed package's own docstring).


In [ ]:
try:
    fork_result = fork_session(initial_session_id, directory=str(DEMO_DIR), title="testing-strategy-fork")
    forked_session_id = fork_result.session_id

    original_messages = get_session_messages(initial_session_id, directory=str(DEMO_DIR))
    forked_messages = get_session_messages(forked_session_id, directory=str(DEMO_DIR))

    print("Original session id:", initial_session_id)
    print("Forked session id:  ", forked_session_id)
    print()
    print(f"Original transcript has {len(original_messages)} messages.")
    print(f"Forked transcript has   {len(forked_messages)} messages (copied at fork time).")
    print()
    print("From here, anything sent to the ORIGINAL session and anything sent to the")
    print("FORKED one would diverge independently -- two branches from one baseline,")
    print("neither able to see what happens in the other. That's the whole mechanism;")
    print("no further real calls are needed to prove the branching itself happened.")
except FileNotFoundError as e:
    forked_session_id = None
    print(f"Could not fork the session yet: {e}")
    print("The transcript file may not be flushed to disk yet -- re-run this cell.")


## 🛠️ Build Exercise — Task 2: A Structured Summary of Findings

**Objective:** distill Task 1's findings into a compact, structured summary
— the exact thing that gets injected into a fresh session later, instead of
dragging along the full stale transcript.

**Why this matters:** this is the difference between "preserve the
knowledge" and "preserve the *baggage*." A summary carries conclusions
without carrying every now-outdated tool result alongside them.


In [ ]:
PRIOR_FINDINGS_SUMMARY = """
Prior analysis of the 10-file codebase found:
- file_05.js: getUserTheme() accesses req.user.profile.settings.theme with
  no null/undefined guards -- a null-pointer risk if any link in that chain
  is missing. (Severity: medium)
- file_08.js: getShippingCity() accesses order.customer.address.shipping.city
  with no guards -- same risk. (Severity: medium)
- file_10.js: getInvoiceContact() accesses invoice.billing.contact.email.address
  with no guards -- same risk. (Severity: medium)
- file_07.js: findUserById() builds a SQL query via string interpolation of
  userId -- a SQL injection risk. (Severity: high, not yet fixed)
- Remaining files (01-04, 06, 09): no significant issues found.
""".strip()

print(PRIOR_FINDINGS_SUMMARY)


## 🛠️ Build Exercise — Task 3: Modify 3 Files for Real

**Objective:** actually fix the three null-pointer risks on disk — real
file changes, not a description of changes.

**Why this matters:** this is what makes Task 1's session's cached `Read`
results genuinely stale a moment from now. No simulation — the bytes on
disk are about to stop matching what the session "remembers."


In [ ]:
FIXED_FILES = {
    "file_05.js": "function getUserTheme(req) {\n  return req?.user?.profile?.settings?.theme;\n}",
    "file_08.js": "function getShippingCity(order) {\n  return order?.customer?.address?.shipping?.city;\n}",
    "file_10.js": "function getInvoiceContact(invoice) {\n  return invoice?.billing?.contact?.email?.address;\n}",
}

for name, new_content in FIXED_FILES.items():
    (DEMO_DIR / name).write_text(new_content, encoding="utf-8")
    print(f"Fixed {name} -- added optional chaining.")

print()
print("These 3 files on disk now genuinely differ from what Task 1's session read.")


## 🛠️ Build Exercise — Task 4: Resume, and Watch for Stale Context

**Objective:** resume the **original** session (not the fork) and ask about
the current state of the three files that just changed.

**Why this matters:** this is the failure mode itself, live. The prompt
below deliberately invites the model to answer from what it already
"knows" — a fair test of the actual risk, not a rigged one. Whether *this
specific* run reproduces the bug or not, the architectural risk it's
demonstrating is real either way (see the note after the output).

This cell makes 1 real API call.


In [ ]:
resume_options = ClaudeAgentOptions(
    cwd=str(DEMO_DIR),
    allowed_tools=["Read", "Glob"],
    resume=initial_session_id,   # <-- continuing the ORIGINAL session, not the fork
)

RESUME_PROMPT = (
    "Based on what you already know from this session, summarize the "
    "current state of file_05.js, file_08.js, and file_10.js, and whether "
    "the null-pointer-risk issues you identified for them are still present."
)

resume_response_text = None

async def run_resume_check():
    global resume_response_text
    async for message in query(prompt=RESUME_PROMPT, options=resume_options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"[assistant] {block.text}")
                elif isinstance(block, ToolUseBlock):
                    print(f"[assistant -> tool_use] {block.name}({block.input})")
        elif isinstance(message, ResultMessage):
            resume_response_text = message.result

await run_resume_check()

print()
print("=== Resumed session's answer ===")
print(resume_response_text)


In [ ]:
still_flags_risk = any(
    kw in (resume_response_text or "").lower()
    for kw in ("still present", "still a risk", "still an issue", "remains", "not yet fixed", "unresolved")
)
mentions_fix = any(
    kw in (resume_response_text or "").lower()
    for kw in ("fixed", "optional chain", "?.", "no longer", "addressed", "was resolved", "have been resolved")
)

print(f"Response suggests the risk is STILL present: {still_flags_risk}")
print(f"Response acknowledges a fix:                 {mentions_fix}")
print()

if still_flags_risk and not mentions_fix:
    print("Stale context reproduced: the session answered from its cached memory of")
    print("the ORIGINAL (broken) code, even though the files on disk were fixed before")
    print("this call was ever made.")
elif mentions_fix:
    print("This run's model chose to re-verify rather than trust cached history --")
    print("a real, honest possible outcome (nothing here forces it to skip re-reading).")
    print("That doesn't remove the architectural risk: relying on 'what the session")
    print("already knows' is only safe when nothing has actually changed, and this")
    print("prompt gave it every opportunity to lean on stale memory instead of checking.")
else:
    print("Inconclusive this run -- inspect the printed response above directly.")


## 🛠️ Build Exercise — Task 5: Fresh Start With Summary Injection

**Objective:** a brand-new session — no `resume`, no shared history — given
Task 2's summary plus an explicit note of which 3 files changed.

**Why this matters:** this is the exam-favored fix, and the only one of the
four practice-scenario options that actually addresses the root cause:
zero stale tool results, prior knowledge preserved, and re-analysis targeted
at just the files that actually changed.

This cell makes 1 real API call.


In [ ]:
fresh_options = ClaudeAgentOptions(
    cwd=str(DEMO_DIR),
    allowed_tools=["Read", "Glob"],
    # No `resume` at all -- this is a genuinely new session.
)

FRESH_START_PROMPT = (
    f"{PRIOR_FINDINGS_SUMMARY}\n\n"
    "The following 3 files have been modified since that analysis: "
    "file_05.js, file_08.js, and file_10.js. Please re-read and re-analyze "
    "ONLY these 3 files to verify whether the previously identified issues "
    "have been resolved, and note any new issues introduced by the changes. "
    "You do not need to re-examine the other files."
)

fresh_response_text = None

async def run_fresh_start():
    global fresh_response_text
    async for message in query(prompt=FRESH_START_PROMPT, options=fresh_options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"[assistant] {block.text}")
                elif isinstance(block, ToolUseBlock):
                    print(f"[assistant -> tool_use] {block.name}({block.input})")
        elif isinstance(message, ResultMessage):
            fresh_response_text = message.result

await run_fresh_start()

print()
print("=== Fresh session's answer ===")
print(fresh_response_text)


## 🛠️ Build Exercise — Task 6: Compare, Side by Side

**Objective:** put Task 4's and Task 5's answers next to each other and
check which one actually reflects the real, current state of the files.

**Why this matters:** "fresh start is better" shouldn't be taken on faith
any more than anything else in this repo — read what each session actually
said about the exact same real files.


In [ ]:
fresh_mentions_fix = any(
    kw in (fresh_response_text or "").lower()
    for kw in ("fixed", "optional chain", "?.", "no longer", "addressed", "was resolved", "have been resolved")
)
fresh_still_flags_risk = any(
    kw in (fresh_response_text or "").lower()
    for kw in ("still present", "still a risk", "still an issue", "remains", "not yet fixed", "unresolved")
)

print("=== Side-by-side ===")
print()
print("[Task 4 -- RESUMED session]")
print(resume_response_text)
print()
print(f"  -> still flags the (now-fixed) risk: {still_flags_risk}   acknowledges a fix: {mentions_fix}")
print()
print("[Task 5 -- FRESH session + summary injection]")
print(fresh_response_text)
print()
print(f"  -> still flags the (now-fixed) risk: {fresh_still_flags_risk}   acknowledges a fix: {fresh_mentions_fix}")
print()

if fresh_mentions_fix and not fresh_still_flags_risk:
    print("The fresh session correctly reflects the current, fixed state of the files.")
if still_flags_risk and not mentions_fix and fresh_mentions_fix:
    print("And the contrast is exactly the module's point: same underlying model, same")
    print("real files, two different session strategies, two different answers -- one")
    print("of them actually correct.")


## ⚠️ Four Traps (one already demonstrated live)

| # | Trap | Why it fails | Fix |
|---|---|---|---|
| 1 | Full re-exploration of all 10 (or 50) files after only 3 changed | Wasteful — repeats work already done on files that didn't change | Targeted re-analysis: name the changed files, let the summary cover the rest |
| 2 | Resuming after files changed | Stale tool results ride along in history — **you just watched this in Task 4** | Fresh start + summary injection (Task 5) |
| 3 | Treating `fork_session` and `resume` as interchangeable | One branches, one continues — mixing them up loses or duplicates work | Fork for divergence, resume for continuation |
| 4 | Using `fork_session` to "handle" stale context | The fork inherits the exact same stale history it branched from | Fresh start, not a fork, fixes staleness |

Trap 2 is Task 4, not a hypothetical — no need to reproduce it again here.
The other three are written below as real code, then commented out.


In [ ]:
# ============================================================
# ❌ TRAP 1 -- Full re-exploration when 3 of 10 files changed
# ============================================================
# Commented out on purpose.
#
# WASTEFUL_PROMPT = (
#     "Forget everything and re-read and re-analyze all 10 files in this "
#     "directory from scratch before continuing."
# )
#
# Why it fails: 7 of the 10 files never changed. Re-reading and re-reasoning
# about all of them repeats work Task 1 already did correctly -- the summary
# from Task 2 already covers those 7 files' findings; only the 3 changed
# ones need fresh eyes.


In [ ]:
# ============================================================
# ❌ TRAP 3 -- fork_session and resume treated as interchangeable
# ============================================================
# Commented out on purpose.
#
# # Trying to "continue" by forking twice:
# fork_a = fork_session(initial_session_id, directory=str(DEMO_DIR))
# fork_b = fork_session(initial_session_id, directory=str(DEMO_DIR))
# # Both fork_a and fork_b start as independent copies of the SAME baseline --
# # neither one is "the continuation." If the goal was just to keep working
# # on one line of investigation, plain `resume=initial_session_id` was what
# # was needed; forking twice here creates two divergent, unrelated branches
# # instead of one continued line of work.
#
# Why it fails: fork_session always branches, regardless of intent. Reach
# for `resume` alone when there's no actual alternative approach to compare
# against -- forking without a real divergence in mind just produces
# orphaned branches nobody continues.


In [ ]:
# ============================================================
# ❌ TRAP 4 -- Using fork_session to "fix" stale context
# ============================================================
# Commented out on purpose.
#
# recovery_attempt = fork_session(initial_session_id, directory=str(DEMO_DIR),
#                                  title="post-fix-recovery")
# # Then resuming the fork, hoping a "fresh branch" solves the staleness:
# broken_options = ClaudeAgentOptions(
#     cwd=str(DEMO_DIR), allowed_tools=["Read", "Glob"],
#     resume=recovery_attempt.session_id,
# )
#
# Why it fails: fork_session COPIES the existing transcript -- stale tool
# results and all. The fork is exactly as stale as the session it branched
# from; nothing about forking removes the outdated Read results sitting in
# its inherited history. Task 5's fresh session (no resume, no fork, just a
# new start plus an injected summary) is the only one of the three options
# that actually has zero stale tool results in it.


## 🎓 Practice Scenario (from the module)

> A developer resumes a Claude Code session after modifying 3 files in a
> 50-file codebase. The agent gives contradictory advice about the modified
> files — recommending changes already made, referencing code that no
> longer exists. What's the most appropriate approach?
>
> - A. Resume the existing session and ask it to re-read the 3 modified files, keeping the rest of the history for continuity
> - B. Start a completely new session with no carried-over context and re-analyze all 50 files from scratch
> - C. Start a fresh session with an injected summary of prior findings, and inform the agent about the specific 3 file changes for targeted re-analysis
> - D. Use `fork_session` to branch the existing session so the file changes can be incorporated in a separate line of work
>
> **Answer: C.** A leaves the stale results sitting in history regardless of
> the fresh read (this is exactly Task 4's failure mode). B throws away
> perfectly good prior knowledge for no reason (Trap 1). D inherits the same
> stale history it branched from (Trap 4) — fork solves divergence, not
> staleness.


## 🏆 Key Takeaways for Exam Prep

1. **Resume only when nothing's changed.** Full history is a feature right
   up until part of it stops being true.
2. **Stale context is insidious, not obvious** — the agent doesn't announce
   that it's reasoning from outdated data; it just quietly contradicts
   reality (or itself).
3. **Summary injection is the professional fix** — knowledge preserved,
   staleness eliminated, by construction.
4. **Targeted beats total.** Naming the 3 files that changed is strictly
   better than re-exploring all 50 (or all 10).
5. **Fork is for divergence, never for remediation** — it copies the
   problem right along with the history.
6. **The exam tests scenario recognition** — did files change? Are you
   exploring alternatives, or continuing one line? Is prior context still
   valid? Each answer points at exactly one of the three strategies.


---

## 🎉 Quick-Fire Recap — See If It Stuck

You named a session, forked it for free, watched a resumed session answer
from stale memory, then watched a fresh session get the same real files
right. Try these from memory first.

**1. In one line: what's actually wrong with resuming after files change?**
> 💡 The old tool results — from before the change — are still sitting in
> the restored history, right alongside anything new, and the model doesn't
> get to just forget them.

**2. Why doesn't "resume, then ask it to re-read the changed files" fully
fix the problem?**
> 💡 The re-read is accurate, but the stale read from earlier in the same
> history is still there too — for anything not directly about those exact
> files, the model can still lean on the outdated version.

**3. Your teammate wants to compare two different refactor strategies from
the same starting analysis. Resume, or fork?**
> 💡 Fork — this is exactly what divergent exploration from a shared
> baseline means. Resume would just keep extending one single line of work.

**4. Someone suggests forking the stale session to "isolate" the file
changes into their own branch. Does that fix anything?**
> 💡 No — `fork_session` copies the existing transcript, stale tool results
> included. The fork is exactly as stale as what it branched from.

**5. In your own Task 4 vs. Task 5 comparison, which session actually got
the current state of `file_05.js`/`file_08.js`/`file_10.js` right?**
> 💡 Whatever your run showed — the fresh session (Task 5) is the one
> architecturally guaranteed to have zero stale results to contradict
> reality with; the resumed session's correctness (if it happened to
> re-verify) depended on the model choosing to double-check, not on the
> session design itself.

**6. 47 of 50 files didn't change. Why not just resume and skip the whole
fresh-start complexity?**
> 💡 Because "47 files didn't change" and "the session's memory of all 50 is
> still accurate" aren't the same claim — the 3 that DID change poison the
> reliability of anything reasoned from that history, even the parts
> touching the other 47.

---

### 🎓 Domain 1 complete.

Seven modules, and you've now covered the full arc: a hand-built agentic
loop, hand-built and SDK-native multi-agent orchestration, hand-built and
hook-based enforcement, how to structure calls so quality doesn't quietly
degrade at scale, and now the session mechanics that make long-running,
resumable agent work actually reliable. Onward to **Domain 2 — Tool Design
& MCP Integration**, starting with **2.1 — Tool Interface Design**.
